In [1]:
# import os
# from sklearn.model_selection import train_test_split

# print("="*50)
# print(" SAFE LONGITUDINAL PATIENT SPLIT")
# print("="*50)

# processed_dir = "/kaggle/input/datasets/jayanth0047/png-spinal-multiple-myeloma-seg/processed_2.5D_dataset" 
# image_dir = os.path.join(processed_dir, "images")
# mask_dir = os.path.join(processed_dir, "masks")

# all_files = sorted(os.listdir(image_dir))

# # 1. Extract the raw IDs (e.g., 'Myel_012_a')
# raw_patient_ids = [f.split('_slice_')[0] for f in all_files]

# # 2. Extract the BASE humans by stripping off '_a' or '_b'
# # This ensures Myel_012_a and Myel_012_b both become just "Myel_012"
# base_humans = list(set([pid.replace('_a', '').replace('_b', '') for pid in raw_patient_ids]))

# print(f"Total Unique Human Patients found: {len(base_humans)} (Matches TCIA Abstract!)")

# # 3. Split the 67 Base Humans 80/20
# train_base, val_base = train_test_split(base_humans, test_size=0.2, random_state=42)

# # 4. Map the slices to the correct fold based on their Base Human ID
# train_files = []
# val_files = []

# for f in all_files:
#     raw_id = f.split('_slice_')[0]
#     base_id = raw_id.replace('_a', '').replace('_b', '')
    
#     if base_id in train_base:
#         train_files.append(f)
#     elif base_id in val_base:
#         val_files.append(f)

# print(f"Training Slices: {len(train_files)}")
# print(f"Validation Slices: {len(val_files)}")

In [2]:
!pip install albumentations segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


In [3]:
import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
import segmentation_models_pytorch as smp
import albumentations as A
from sklearn.model_selection import train_test_split

print("\n" + "="*50)
print(" STAGE 1: SAFE LONGITUDINAL PATIENT SPLIT")
print("="*50)

# --- DIRECTORIES ---
processed_dir = "/kaggle/input/datasets/jayanth0047/png-spinal-multiple-myeloma-seg/processed_2.5D_dataset" 
image_dir = os.path.join(processed_dir, "images")
mask_dir = os.path.join(processed_dir, "masks")

all_files = sorted(os.listdir(image_dir))

# Extract BASE humans to prevent leakage
raw_patient_ids = [f.split('_slice_')[0] for f in all_files]
base_humans = list(set([pid.replace('_a', '').replace('_b', '') for pid in raw_patient_ids]))
print(f"Total Unique Human Patients found: {len(base_humans)}")

# 80/20 Patient Split
train_base, val_base = train_test_split(base_humans, test_size=0.2, random_state=42)

train_files, val_files = [], []
for f in all_files:
    base_id = f.split('_slice_')[0].replace('_a', '').replace('_b', '')
    if base_id in train_base: train_files.append(f)
    elif base_id in val_base: val_files.append(f)

print(f"Training Slices: {len(train_files)} | Validation Slices: {len(val_files)}")

print("\n" + "="*50)
print(" STAGE 2: PYTORCH DATALOADERS")
print("="*50)

class TCIA_25D_Dataset(Dataset):
    def __init__(self, file_list, img_dir, msk_dir, is_train=False):
        self.file_list = file_list
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        self.is_train = is_train
        
        # Lightweight spatial augmentations
        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5)
        ])

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]
        img = cv2.imread(os.path.join(self.img_dir, filename), cv2.IMREAD_COLOR)
        mask_img = cv2.imread(os.path.join(self.msk_dir, filename), cv2.IMREAD_COLOR)
        
        spine_mask = (mask_img[:, :, 0] > 127).astype(np.uint8)
        lesion_mask = (mask_img[:, :, 1] > 127).astype(np.uint8)
        
        if self.is_train:
            augmented = self.aug(image=img, masks=[spine_mask, lesion_mask])
            img, spine_mask, lesion_mask = augmented['image'], augmented['masks'][0], augmented['masks'][1]
            
        img = img.astype(np.float32) / 255.0
        spine_mask, lesion_mask = spine_mask.astype(np.float32), lesion_mask.astype(np.float32)
        
        img_tensor = torch.tensor(img).permute(2, 0, 1)
        mask_tensor = torch.stack([torch.tensor(spine_mask), torch.tensor(lesion_mask)], dim=0)
        
        return img_tensor, mask_tensor

# train_loader = DataLoader(TCIA_25D_Dataset(train_files, image_dir, mask_dir, True), batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
# val_loader = DataLoader(TCIA_25D_Dataset(val_files, image_dir, mask_dir, False), batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
# Dropped to 8 to fit safely in the T4 VRAM
train_loader = DataLoader(TCIA_25D_Dataset(train_files, image_dir, mask_dir, True), batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(TCIA_25D_Dataset(val_files, image_dir, mask_dir, False), batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

print("\n" + "="*50)
print(" STAGE 3: LOSS & ARCHITECTURE")
print("="*50)

def dice_coeff(pred, target, smooth=1e-6):
    pred_flat, target_flat = pred.contiguous().view(-1), target.contiguous().view(-1)
    return (2. * (pred_flat * target_flat).sum() + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

def tversky_loss(pred, target, alpha=0.3, beta=0.7, smooth=1e-6):
    pred_flat, target_flat = pred.contiguous().view(-1), target.contiguous().view(-1)
    tp = (pred_flat * target_flat).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    fn = ((1 - pred_flat) * target_flat).sum()
    return 1 - ((tp + smooth) / (tp + alpha * fp + beta * fn + smooth))

class RobustLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, pred_logits, target):
        bone_gt, lesion_gt = target[:, 0, ...], target[:, 1, ...]
        pred_probs = torch.sigmoid(pred_logits)
        
        bce_bone = F.binary_cross_entropy_with_logits(pred_logits[:, 0, ...], bone_gt, reduction='none')
        bone_focal = (0.75 * (1 - torch.exp(-bce_bone)) ** 2.0 * bce_bone).mean() 
        bce_lesion = F.binary_cross_entropy_with_logits(pred_logits[:, 1, ...], lesion_gt, reduction='none')
        lesion_focal = (0.75 * (1 - torch.exp(-bce_lesion)) ** 3.0 * bce_lesion).mean()
        
        bone_dice = 1 - dice_coeff(pred_probs[:, 0, ...], bone_gt)
        # CRITICAL: Heavy False Positive Penalty (Alpha=0.75)
        lesion_tversky = tversky_loss(pred_probs[:, 1, ...], lesion_gt, 0.75, 0.25)
        
        return (bone_focal + bone_dice) + 3.0 * (lesion_focal + lesion_tversky)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=2, decoder_attention_type='scse').to(device)

criterion = RobustLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
scaler = GradScaler('cuda')

best_hard_dice = 0.0
save_path = "/kaggle/working/tcia_resnet34_best.pth"
epochs = 20  # You can increase this if you want it to run longer overnight

print("\n" + "="*50)
print(" STAGE 4: TRAINING RUN (WITH GRADIENT ACCUMULATION)")
print("="*50)

accumulation_steps = 4 # 8 batch_size * 4 steps = 32 effective batch size

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad(set_to_none=True) # Zero out at the very beginning
    
    for i, (images, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]", leave=False)):
        images, masks = images.to(device), masks.to(device)
        
        with autocast('cuda'): 
            loss = criterion(model(images), masks)
            loss = loss / accumulation_steps # Normalize loss for accumulation
            
        scaler.scale(loss).backward()
        
        # Only update the weights after 'accumulation_steps' batches have been processed
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True) # Reset for the next accumulation cycle

    # --- VALIDATION LOOP ---
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validating", leave=False):
            images, masks = images.to(device), masks.to(device)
            with autocast('cuda'): # Use AMP in validation to save memory too!
                logits = model(images)
                probs = torch.sigmoid(logits)[:, 1, ...] 
            
            all_probs.append(probs.cpu().reshape(-1))
            all_targets.append(masks[:, 1, ...].cpu().reshape(-1))
            
    flat_probs = torch.cat(all_probs)
    flat_targets = torch.cat(all_targets)
    
    val_soft_dice = dice_coeff(flat_probs, flat_targets).item()
    val_hard_dice = dice_coeff((flat_probs > 0.5).float(), flat_targets).item()
    
    scheduler.step(val_soft_dice)
    
    if val_hard_dice > best_hard_dice:
        best_hard_dice = val_hard_dice
        torch.save(model.state_dict(), save_path)
        
    print(f"Epoch {epoch+1} -> Val Soft Dice: {val_soft_dice:.4f} | Val Hard Dice (@0.5): {val_hard_dice:.4f}")

print("\n" + "="*50)
print(f"🔥 TRAINING COMPLETE! Peak Hard Dice: {best_hard_dice:.4f} 🔥")
print("="*50)

# =====================================================================
# VISUAL AUTOPSY SCRIPT (Runs immediately after training)
# =====================================================================
print("\nGenerating Visual Autopsy...")
model.load_state_dict(torch.load(save_path))
model.eval()

# Sample 3 slices from the Validation Set that ACTUALLY contain a tumor
val_dataset = val_loader.dataset
tumor_indices = []
for i in range(min(2000, len(val_dataset))): # Scan first 2000 to save time
    _, mask = val_dataset[i]
    if mask[1].sum() > 0: tumor_indices.append(i)

sampled_indices = random.sample(tumor_indices, min(3, len(tumor_indices)))
fig, axes = plt.subplots(len(sampled_indices), 4, figsize=(20, 5 * len(sampled_indices)))
if len(sampled_indices) == 1: axes = [axes]
fig.suptitle("TCIA 40keV CLINICAL DATA AUTOPSY", fontsize=16, fontweight='bold', y=1.02)

for i, idx in enumerate(sampled_indices):
    img_tensor, mask_tensor = val_dataset[idx]
    img_batch = img_tensor.unsqueeze(0).to(device)
    gt_mask = mask_tensor[1].numpy()
    
    with torch.no_grad():
        pred_prob = torch.sigmoid(model(img_batch))[0, 1].cpu().numpy()
        
    pred_mask = (pred_prob > 0.5).astype(np.uint8)
    
    error_map = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    error_map[np.logical_and(pred_mask == 1, gt_mask == 1)] = [0, 255, 0]   # GREEN: Hit
    error_map[np.logical_and(pred_mask == 1, gt_mask == 0)] = [0, 0, 255]   # BLUE: False Alarm
    error_map[np.logical_and(pred_mask == 0, gt_mask == 1)] = [255, 0, 0]   # RED: Miss
    
    # We display the middle channel (Green/Index 1) which represents the 'Current' Z-slice
    raw_img = img_tensor[1].numpy() 
    
    filename = val_dataset.file_list[idx]
    axes[i][0].imshow(raw_img, cmap='gray'); axes[i][0].set_title(f"CT: {filename}"); axes[i][0].axis('off')
    axes[i][1].imshow(gt_mask, cmap='hot'); axes[i][1].set_title("Ground Truth"); axes[i][1].axis('off')
    axes[i][2].imshow(pred_prob, cmap='inferno'); axes[i][2].set_title("Prob Map (Thresh: 0.50)"); axes[i][2].axis('off')
    axes[i][3].imshow(error_map); axes[i][3].set_title("Errors (G=Hit, R=Miss, B=False Alarm)"); axes[i][3].axis('off')

plt.tight_layout()
plt.show()


 STAGE 1: SAFE LONGITUDINAL PATIENT SPLIT
Total Unique Human Patients found: 67
Training Slices: 53396 | Validation Slices: 12619

 STAGE 2: PYTORCH DATALOADERS

 STAGE 3: LOSS & ARCHITECTURE


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]


 STAGE 4: TRAINING RUN (WITH GRADIENT ACCUMULATION)
